Tomando las dos bases de datos que sirven para entrenar el modelo incident_event_log.csv y ITSM_data.csv idemtificamos que tienen similitudes pero usan formatos diferentes como el INC00005 para identificar los tickets, y el archvio ITSM_data.csv usa IM00004.Tienen diferentes letras. Entonces la solucion planteada es cruzar estas dos tablas usando el numero dle incidente.Esto nos permitira tener una sola fila de vida detallada del ticket de incident_event_log.csv junto con la informacion del sistema afectado (CI_name, wbs de  ITSM_data.csv ).

Leemos los dos archivos que vamos a cruzar

In [2]:
import pandas as pd

df_incidentes = pd.read_csv("incident_event_log.csv")
df_itsm = pd.read_csv("ITSM_data.csv", low_memory=False)

print("Datos cargados correctamente.")

Datos cargados correctamente.


1) Sacamos solo los numeros de los formatos INC0000045 y IM0000004
2) Limpiamos nulos y convertimos a numero entero

In [3]:

df_incidentes['ticket_id'] = df_incidentes['number'].str.extract(r'(\d+)')[0].astype(float)
df_itsm['ticket_id'] = df_itsm['Incident_ID'].str.extract(r'(\d+)')[0].astype(float)

df_incidentes = df_incidentes.dropna(subset=['ticket_id'])
df_itsm = df_itsm.dropna(subset=['ticket_id'])

df_incidentes['ticket_id'] = df_incidentes['ticket_id'].astype(int)
df_itsm['ticket_id'] = df_itsm['ticket_id'].astype(int)

print("IDs limpios")

IDs limpios


Nos quedamos solo con la ultima actualizacion de cada ticket en la base de incidentes.La razon por la que hacemos ese paso es porque el archivo incident_event_log.csv no es una lista de tickets, sino un historial de eventos.

In [ ]:

df_incidentes_final = df_incidentes.sort_values(by=['ticket_id', 'sys_mod_count']).drop_duplicates(subset='ticket_id', keep='last')

print(f"Tickets únicos listos: {len(df_incidentes_final)}")

Tickets únicos listos: 24918


In [4]:
# Cruzamos las bases de datos usando nuestro nuevo ticket_id
dataset_final = pd.merge(df_incidentes_final, df_itsm, on='ticket_id', how='inner', suffixes=('_inc', '_itsm'))

# Guardamos el resultado en la misma carpeta
dataset_final.to_csv("dataset_combinado_final.csv", index=False)

print(f"¡Combinación exitosa! Filas: {dataset_final.shape[0]}, Columnas: {dataset_final.shape[1]}")

¡Combinación exitosa! Filas: 24483, Columnas: 62


In [ ]:

dataset = pd.read_csv("dataset_combinado_final.csv", low_memory=False)

nulos_por_columna = dataset.isnull().sum()


columnas_con_nulos = nulos_por_columna[nulos_por_columna > 0].sort_values(ascending=False)

print("=== COLUMNAS CON DATOS FALTANTES (NULOS) ===")
print(columnas_con_nulos.head(15)) # Mostramos el Top 15
print(f"\nTotal de filas en el dataset: {len(dataset)}")

=== COLUMNAS CON DATOS FALTANTES (NULOS) ===
Related_Change                24209
No_of_Related_Changes         24209
No_of_Related_Incidents       23920
Reopen_Time                   23239
Resolved_Time                   974
Priority                        660
Closure_Code                    263
CI_Cat                           85
CI_Subcat                        85
No_of_Related_Interactions       71
dtype: int64

Total de filas en el dataset: 24483


In [ ]:
print("=== TIPOS DE DATOS EN NUESTRO DATASET ===")
tipos_datos = dataset.dtypes.value_counts()
print(tipos_datos)


columnas_texto = dataset.select_dtypes(include=['object']).columns
print(f"\nTenemos {len(columnas_texto)} columnas de texto que luego tendremos que convertir a numeros.")

=== TIPOS DE DATOS EN NUESTRO DATASET ===
str        47
float64     6
int64       5
bool        4
Name: count, dtype: int64

Tenemos 47 columnas de texto que luego tendremos que convertir a números.


C:\Users\jesus\AppData\Local\Temp\ipykernel_14324\4179542112.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  columnas_texto = dataset.select_dtypes(include=['object']).columns


In [ ]:
import numpy as np

dataset.replace('?', np.nan, inplace=True)


nulos_totales = dataset.isnull().sum()
porcentaje_nulos = (nulos_totales / len(dataset)) * 100


columnas_afectadas = porcentaje_nulos[porcentaje_nulos > 0].sort_values(ascending=False)

print("=== PORCENTAJE REAL DE DATOS FALTANTES ===")

for columna, porcentaje in columnas_afectadas.items():
    print(f"{columna}: {porcentaje:.2f}%")

=== PORCENTAJE REAL DE DATOS FALTANTES ===
caused_by: 99.99%
vendor: 99.94%
cmdb_ci: 99.78%
rfc: 99.29%
No_of_Related_Changes: 98.88%
Related_Change: 98.88%
problem_id: 98.51%
No_of_Related_Incidents: 97.70%
Reopen_Time: 94.92%
sys_created_at: 46.12%
sys_created_by: 46.12%
u_symptom: 23.48%
assignment_group: 8.76%
resolved_at: 6.28%
Resolved_Time: 3.98%
opened_by: 2.89%
assigned_to: 2.84%
Priority: 2.70%
Closure_Code: 1.07%
closed_code: 0.43%
resolved_by: 0.36%
CI_Cat: 0.35%
CI_Subcat: 0.35%
No_of_Related_Interactions: 0.29%
subcategory: 0.03%
category: 0.03%
location: 0.02%
caller_id: 0.01%


In [ ]:

limite_minimo_datos = int(len(dataset) * 0.60)


dataset_limpio = dataset.dropna(thresh=limite_minimo_datos, axis=1).copy()

columnas_eliminadas = dataset.shape[1] - dataset_limpio.shape[1]

print(f"Teníamos {dataset.shape[1]} columnas.")
print(f"Ahora tenemos {dataset_limpio.shape[1]} columnas.")
print(f"¡Eliminamos {columnas_eliminadas} columnas basura de un plumazo!")

Teníamos 62 columnas.
Ahora tenemos 51 columnas.
¡Eliminamos 11 columnas basura de un plumazo!


In [ ]:

columnas_texto = dataset_limpio.select_dtypes(include=['object']).columns
columnas_numericas = dataset_limpio.select_dtypes(include=['float64', 'int64', 'int32']).columns


dataset_limpio[columnas_texto] = dataset_limpio[columnas_texto].fillna("Desconocido")


dataset_limpio[columnas_numericas] = dataset_limpio[columnas_numericas].fillna(dataset_limpio[columnas_numericas].median())


nulos_totales_ahora = dataset_limpio.isnull().sum().sum()
print(f"Total de datos nulos en el dataset ahora mismo: {nulos_totales_ahora}")

Total de datos nulos en el dataset ahora mismo: 0


C:\Users\jesus\AppData\Local\Temp\ipykernel_14324\2745560311.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  columnas_texto = dataset_limpio.select_dtypes(include=['object']).columns


In [ ]:

import pandas as pd
from sklearn.preprocessing import LabelEncoder


dataset_encoded = dataset_limpio.copy()


le = LabelEncoder()


columnas_texto = dataset_encoded.select_dtypes(include=['object']).columns

print("Traduciendo las siguientes columnas a numeros:")

for columna in columnas_texto:
    print(f"- {columna}")
  
    dataset_encoded[columna] = le.fit_transform(dataset_encoded[columna].astype(str))

print("\n¡Traducción completada con éxito!")

dataset_encoded.head()

C:\Users\jesus\AppData\Local\Temp\ipykernel_14324\2945129159.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  columnas_texto = dataset_encoded.select_dtypes(include=['object']).columns


Traduciendo las siguientes columnas a números:
- number
- incident_state
- caller_id
- opened_by
- opened_at
- sys_updated_by
- sys_updated_at
- contact_type
- location
- category
- subcategory
- u_symptom
- impact
- urgency
- priority
- assignment_group
- assigned_to
- notify
- closed_code
- resolved_by
- resolved_at
- closed_at
- CI_Name
- CI_Cat
- CI_Subcat
- WBS
- Incident_ID
- Status
- Impact
- Category
- KB_number
- Alert_Status
- Open_Time
- Resolved_Time
- Close_Time
- Handle_Time_hrs
- Closure_Code
- Related_Interaction

¡Traducción completada con éxito!


,number,incident_state,active,reassignment_count,reopen_count,sys_mod_count,made_sla,caller_id,opened_by,opened_at,...,KB_number,Alert_Status,No_of_Reassignments,Open_Time,Resolved_Time,Close_Time,Handle_Time_hrs,Closure_Code,No_of_Related_Interactions,Related_Interaction
0,0,0,False,0,0,4,True,1442,158,12733,...,613,0,7.0,13998,10956,11282,7780,11,1.0,2
1,1,0,False,1,0,8,True,1442,99,12734,...,251,0,4.0,15983,6564,6739,9415,7,1.0,3
2,2,0,False,0,0,6,True,3497,158,12735,...,523,0,1.0,10360,2208,2307,10252,7,3.0,0
3,3,0,False,0,0,3,True,3565,28,12736,...,1091,0,0.0,12048,7478,7674,7394,5,1.0,4
4,4,0,False,1,0,7,False,2843,28,12737,...,351,0,2.0,16213,18329,18910,14471,5,1.0,5


In [ ]:
from sklearn.model_selection import train_test_split

y = dataset_encoded['priority']


columnas_trampa = ['priority', 'Priority', 'ticket_id', 'impact', 'urgency', 'Impact', 'Urgency', 'u_priority_confirmation']
X_real = dataset_encoded.drop(columns=columnas_trampa, errors='ignore')

X_train, X_test, y_train, y_test = train_test_split(X_real, y, test_size=0.20, random_state=42)

print("Datos separados")

¡Datos separados correctamente y sin trampas!


In [ ]:
from sklearn.ensemble import RandomForestClassifier

print("Entrenando al modelo solo con categoria, ubicacion, sistemas, etc...")

modelo_ia = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)


modelo_ia.fit(X_train, y_train)

print("Entrenamiento finalizado")

Entrenando al modelo solo con categoría, ubicación, sistemas, etc...
¡Entrenamiento finalizado!


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

print("Haciendo que la IA resuelva")

predicciones = modelo_ia.predict(X_test)


calificacion = accuracy_score(y_test, predicciones) * 100

print(f"\n=== RESULTADOS DEL EXAMEN REAL ===")
print(f"Precision REAL (Accuracy): {calificacion:.2f}%")


Haciendo que la IA resuelva el examen difícil...

=== RESULTADOS DEL EXAMEN REAL ===
Precisión REAL (Accuracy): 95.49%
